In [30]:
import pandas as pd
from config import *
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import numpy as np
from config import *

In [ ]:
path = WOLT_DATA_DIR / 'copenhagen'  
venues = pd.read_csv(path / 'venues.csv')
items = pd.read_csv(path / 'items.csv')
item_ingredients = pd.read_csv(path / 'item_ingredients.csv')
ingredients_raw = pd.read_csv(path / 'ingredients.csv').dropna()
ingredients_food = ingredients_raw[ingredients_raw['is_ingredient'] == 1.0]


In [32]:
ingredients_food.head()

,ingredient_raw,n,ingredient_verified,is_ingredient
0,tomat,19600,tomat,1.0
1,ost,18226,ost,1.0
2,salat,15671,salat,1.0
3,agurk,12524,agurk,1.0
4,løg,12093,løg,1.0


In [33]:
ingredients_food.drop(columns=['n', 'ingredient_raw', 'is_ingredient'], inplace=True)
ingredients_food.drop_duplicates(inplace=True)

In [4]:
dishes_ingredients = (
    item_ingredients
    .merge(ingredients_food[["ingredient_raw", "ingredient_verified"]], on="ingredient_raw")
    .rename(columns={"ingredient_verified": "ingredient"})[["item_id", "ingredient"]]
    .drop_duplicates() 
    .reset_index(drop=True)
)

In [5]:
pairs = (
    dishes_ingredients
    .merge(dishes_ingredients, on="item_id", suffixes=("_a", "_b"))
    .rename(columns={"ingredient_a": "src", "ingredient_b": "trg"})
)
pairs = pairs[pairs["src"] < pairs["trg"]]   # each pair once, no self-loops
pairs.head()

,item_id,src,trg
1,62165417a1899fbdc42e2beb,avocado,tun
2,62165417a1899fbdc42e2beb,avocado,laks
3,62165417a1899fbdc42e2beb,avocado,reje
9,62165417a1899fbdc42e2beb,laks,tun
11,62165417a1899fbdc42e2beb,laks,reje


In [6]:
edges = (
    pairs.groupby(["src", "trg"])
    .size()
    .reset_index(name="weight")
    .sort_values(by='weight', ascending=False)
)

In [7]:
edges.to_csv(NETWORK_DATA_DIR / "edges.csv", index=False)

# Network

In [8]:
import backboning

table, nnodes, nnedges = backboning.read(NETWORK_DATA_DIR / "edges.csv", column_of_interest="weight", triangular_input=True, consider_self_loops=False, undirected=True, sep=",")
nc_table = backboning.noise_corrected(table)
nc_backbone = backboning.thresholding(nc_table, 100) # what should the threshold value be ?
backboning.write(nc_backbone, "ingredient_network", "nc", NETWORK_DATA_DIR)

Calculating NC score...
